# 🛡️ Fake News & Misinformation Detection Classifier
### End-to-End NLP Pipeline: Traditional ML vs. Transformer Fine-Tuning

This notebook implements an end-to-end Machine Learning and Natural Language Processing project comparing three distinct paradigms:
1. **TF-IDF + Logistic Regression** (High-speed linear baseline)
2. **TF-IDF + Random Forest** (Non-linear ensemble)
3. **Fine-Tuned DistilBERT** (State-of-the-art transformer with contextual self-attention)

---


In [ ]:
# 1. Imports and Reproducibility Setup
import os
import sys
import json
import torch
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Set styles & seed
sns.set_theme(style="whitegrid", palette="muted")
np.random.seed(42)
torch.manual_seed(42)

# Ensure project root is in path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("PyTorch CUDA Available:", torch.cuda.is_available())


## 2. Dataset Ingestion & Exploratory Data Analysis (EDA)
We load the verified benchmark dataset containing news titles, article text, and ground truth labels (`REAL`: 0, `FAKE`: 1).


In [ ]:
from src.data_loader import prepare_pipeline

# Execute data acquisition, cleaning, EDA, and stratified train/val/test splits
train_df, val_df, test_df = prepare_pipeline()

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")
display(train_df.head(3))


### Visualizing Class & Word Count Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class Distribution
train_df['label'].value_counts().plot(kind='bar', color=['#2E7D32', '#C62828'], ax=axes[0])
axes[0].set_title("Training Set Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_xticklabels(["REAL (0)", "FAKE (1)"], rotation=0)
axes[0].set_ylabel("Count")

# Word Count Distribution
sns.histplot(data=train_df[train_df['word_count'] < 2000], x='word_count', hue='label', palette={0: '#2E7D32', 1: '#C62828'}, ax=axes[1], element="step")
axes[1].set_title("Article Word Count by Label", fontsize=13, fontweight="bold")
axes[1].legend(["FAKE", "REAL"])

plt.tight_layout()
plt.show()


## 3. Text Preprocessing Pipelines
We employ separate preprocessing logic:
- **Traditional Preprocessing:** Lowercasing, HTML/URL stripping, punctuation removal, stopword removal, WordNet lemmatization.
- **Transformer Preprocessing:** Preserving casing, punctuation, and contextual flow for DistilBERT self-attention.


In [ ]:
from src.preprocessing import preprocess_traditional, preprocess_transformer, prepare_dataset_features

sample_text = "<b>BREAKING:</b> Secret miracle cure revealed at http://cure.com/now! Doctors are amazed in 2026."
print("Original Text      :", sample_text)
print("Traditional Clean  :", preprocess_traditional(sample_text))
print("Transformer Clean  :", preprocess_transformer(sample_text))


## 4. Model 1: TF-IDF + Logistic Regression

In [ ]:
from src.train_logistic import run_pipeline as run_logistic

lr_metrics = run_logistic()


## 5. Model 2: TF-IDF + Random Forest Classifier

In [ ]:
from src.train_random_forest import run_pipeline as run_rf

rf_metrics = run_rf()


## 6. Model 3: Fine-Tuning DistilBERT (Transformers + PyTorch)

In [ ]:
from src.train_distilbert import run_pipeline as run_distilbert

db_metrics = run_distilbert()


## 7. Model Benchmark Comparison & Evaluation

In [ ]:
from src.evaluate import evaluate_all_models

comparison_results = evaluate_all_models()


## 8. Model Explainability & Saliency Analysis
Analyzing token-level contributions for individual articles across the models.


In [ ]:
from src.explainability import explain_prediction

test_article = (
    "SHOCKING BREAKING DISCOVERY: Doctors are furious after a whistleblower leaked a secret miracle cure that Big Pharma "
    "has been hiding from the public for decades! Mainstream media is censoring this."
)

print("--- Logistic Regression Explanation ---")
exp_lr = explain_prediction(test_article, model_choice="Logistic Regression")
print(f"Prediction: {exp_lr['prediction']} (Confidence: {exp_lr['confidence']}%)")
print("Top Contributory Tokens:", exp_lr['top_contributing_words'][:6])

print("\n--- DistilBERT Explanation ---")
exp_db = explain_prediction(test_article, model_choice="DistilBERT")
print(f"Prediction: {exp_db['prediction']} (Confidence: {exp_db['confidence']}%)")
print("Top Contributory Tokens:", exp_db['top_contributing_words'][:6])
